# Fase 3 - Preparacao dos Dados (Pacote A)

Matriz transacional one-hot: subset com vitimas, alvo `desfecho`, itens `ctx_*`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import (
    ANOS_OCORRENCIA,
    COLUNAS_CONTEXTO,
    DATA_DIR,
    DADOS_DIR,
    FIGURAS_DIR,
    FILTRO_UF,
    MIN_SUPPORT,
    PROCESSED_DIR,
    TABELAS_DIR,
)

plt.rcParams["figure.figsize"] = (11, 5)
sns.set_style("whitegrid")
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

## 3.1 Pipeline

In [2]:
from src.data_loading import carregar_anos
from src.preparation import preparar_pipeline

df_raw, _ = carregar_anos(ignorar_ausentes=True)
df_onehot, df_meta, info = preparar_pipeline(df_raw)
for k, v in info.items():
    print(f"  {k}: {v}")

[OK] Anos carregados: [2023, 2024, 2025, 2026] | Registros: 30,858
[OK] Preparacao: 26,899 transacoes | Itens: 455 -> 78
     Fatal: 7.83% | Ferido: 92.17%
  n_registros: 26899
  n_itens_total: 455
  n_itens_filtrado: 78
  min_freq: 0.005
  max_freq: 0.99
  colunas_contexto: ['condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'causa_acidente', 'tipo_acidente', 'fase_dia', 'fim_de_semana']
  pct_fatal: 7.83
  pct_ferido: 92.17
  design: pacote_a_contexto_para_desfecho
  anos: [2023, 2024, 2025, 2026]


## 3.2 Validacao

In [3]:
ctx_cols = [c for c in df_onehot.columns if c.startswith("ctx_")]
alvo_cols = [c for c in df_onehot.columns if c.startswith("desfecho_")]
proibidos = [c for c in df_onehot.columns if any(x in c for x in ("gravidade", "classificacao", "faixa_horaria"))]
assert not proibidos
print(f"Contexto: {len(ctx_cols)} | Alvo: {alvo_cols} | Transacoes: {len(df_onehot):,}")
display(df_onehot.sum().sort_values(ascending=False).head(12).to_frame("freq_abs"))

Contexto: 76 | Alvo: ['desfecho_Fatal', 'desfecho_Ferido'] | Transacoes: 26,899


,freq_abs
desfecho_Ferido,24793
ctx_uso_solo_Não,18801
ctx_fim_de_semana_Não,18152
ctx_condicao_metereologica_Céu_Claro,16341
ctx_fase_dia_Pleno_dia,15807
ctx_tipo_pista_Simples,13708
ctx_tipo_pista_Dupla,12051
ctx_tracado_via_Reta,11444
ctx_fim_de_semana_Sim,8747
ctx_fase_dia_Plena_Noite,8326


## 3.3 Amostra

In [4]:
display(df_meta.head(8))
idx = df_onehot[df_onehot["desfecho_Fatal"]].index[0]
print("Itens ativos (exemplo fatal):", df_onehot.columns[df_onehot.loc[idx]].tolist())

,id,ano,desfecho,uso_solo
0,496659.0,2023,Ferido,Não
1,496795.0,2023,Ferido,Não
2,496807.0,2023,Ferido,Não
3,496811.0,2023,Ferido,Não
4,496818.0,2023,Fatal,Não
5,496862.0,2023,Fatal,Não
6,496904.0,2023,Ferido,Não
7,496929.0,2023,Ferido,Não


Itens ativos (exemplo fatal): ['ctx_causa_acidente_Reação_tardia_ou_ineficiente_do_condutor', 'ctx_condicao_metereologica_Garoa/Chuvisco', 'ctx_fase_dia_Pleno_dia', 'ctx_fim_de_semana_Não', 'ctx_tipo_acidente_Colisão_lateral_sentido_oposto', 'ctx_tipo_pista_Simples', 'ctx_tracado_via_Curva;Declive', 'ctx_uso_solo_Não', 'desfecho_Fatal']


## 3.4 Artefatos

In [5]:
for p in [PROCESSED_DIR / "df_limpo.pkl", PROCESSED_DIR / "transacional.pkl",
          PROCESSED_DIR / "transacional_meta.pkl", PROCESSED_DIR / "preparacao_metadata.json"]:
    print(f"  {'[OK]' if p.exists() else '[--]'} {p.relative_to(ROOT)}")

  [OK] data\processed\df_limpo.pkl
  [OK] data\processed\transacional.pkl
  [OK] data\processed\transacional_meta.pkl
  [OK] data\processed\preparacao_metadata.json
